<a href="https://colab.research.google.com/github/abrandonwang/text-to-sound/blob/main/latent_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/text-to-sound/"
CKPT = DRIVE + "checkpoints/"
DATA = DRIVE + "data/"

In [ ]:
!test -d BigVGAN || git clone -q https://github.com/NVIDIA/BigVGAN
!sed -i 's/proxies: Optional\[Dict\],/proxies: Optional[Dict] = None,/; s/resume_download: bool,/resume_download: bool = False,/' BigVGAN/bigvgan.py
import sys; sys.path.insert(0, "BigVGAN")
import bigvgan
import glob, math, copy, random, os, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import Audio, display
from transformers import ClapModel, ClapProcessor
%matplotlib inline

device = "cuda"
print(torch.cuda.get_device_name())

vocoder = bigvgan.BigVGAN.from_pretrained("nvidia/bigvgan_v2_22khz_80band_256x", use_cuda_kernel = False)
vocoder.remove_weight_norm()
vocoder = vocoder.eval().to(device)

clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

MIN, MAX = np.load(DATA + "mel_stats.npy")

In [ ]:
class Autoencoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = nn.Sequential(
      nn.Conv2d(1, 64, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(64, 64, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(64, 128, 3, 2, 1),
      nn.SiLU(),
      nn.Conv2d(128, 128, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(128, 256, 3, 2, 1),
      nn.SiLU(),
      nn.Conv2d(256, 256, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(256, 8, 3, 1, 1),
    )
    self.decoder = nn.Sequential(
        nn.Conv2d(4, 256, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(256, 256, 3, 1, 1),
        nn.SiLU(),
        nn.Upsample(scale_factor = 2),
        nn.Conv2d(256, 128, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(128, 128, 3, 1, 1),
        nn.SiLU(),
        nn.Upsample(scale_factor = 2),
        nn.Conv2d(128, 64, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(64, 64, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(64, 1, 3, 1, 1),
    )
  def forward(self, x):
    mu, logvar = self.encoder(x).chunk(2, dim = 1)
    z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
    return self.decoder(z), mu, logvar

model = Autoencoder().to(device)
model.load_state_dict(torch.load(CKPT + "vae_gan2.pt"))

In [ ]:
SCALE = 0.6863   # Clotho-only scale; only used by unpad() below, replaced after loading

# Clotho: train split only, so val stays unseen
meta = pd.read_csv(DATA + "meta.csv")
tr = (meta.split == "train").values
c_lat = np.load(DATA + "latents.npy")[tr]
c_aud = np.load(DATA + "clap_audio.npy")[tr]
c_txt = np.load(DATA + "clap_text.npy")[tr]

# FSD50K: all 21 shards
shards = [np.load(f) for f in sorted(glob.glob(DATA + "fsd/shard_*.npz"))]
f_lat = np.concatenate([s["latents"] for s in shards])
f_aud = np.concatenate([s["clap_audio"] for s in shards])
f_txt = np.concatenate([s["clap_text"] for s in shards])

# Short FSD50K clips were padded with digital silence, which this VAE encodes as huge
# values (mean ~+4). Keep only the real-sound columns and repeat them to fill the crop.
def unpad(z, thr = 2.5, margin = 1, min_cols = 4):
    col = z.astype(np.float32).mean(axis = (0, 1)) * SCALE     # mean per time column
    sound = np.nonzero(col < thr)[0]
    if len(sound) == 64:
        return z                                                # no padding
    end = (sound.max() + 1 - margin) if len(sound) else 0
    if end < min_cols:
        return None                                             # under ~0.2 s of sound: drop
    return np.tile(z[..., :end], (1, 1, -(-64 // end)))[..., :64]

fixed = [unpad(z) for z in f_lat]
keep = np.array([z is not None for z in fixed])
print("fsd padded:", sum(f is not z for f, z in zip(fixed, f_lat) if f is not None), "| dropped:", (~keep).sum())
print("fsd latent std before:", (f_lat.astype(np.float32) * SCALE).std())
f_lat = np.stack([z for z in fixed if z is not None])
f_aud, f_txt = f_aud[keep], f_txt[keep]
print("fsd latent std after:", (f_lat.astype(np.float32) * SCALE).std(), "(clotho:", (c_lat.astype(np.float32) * SCALE).std(), ")")

lat = np.concatenate([c_lat, f_lat]).astype(np.float16)
aud = np.concatenate([c_aud, f_aud]).astype(np.float32)
txt = np.concatenate([c_txt, f_txt]).astype(np.float32)
print("clotho", len(c_lat), "| fsd", len(f_lat), "| total", len(lat))

# LDM/AudioLDM "scale_by_std": scale the latents the diffusion model sees to std 1
SCALE = float(1 / lat[::10].astype(np.float32).std())
print("latent scale:", SCALE, "(Clotho-only was 0.6863)")

class LatentDataset(Dataset):
    def __init__(self, lat, aud, txt):
        self.lat, self.aud, self.txt = lat, aud, txt
    def __len__(self):
        return len(self.lat)
    def __getitem__(self, i):
        z = torch.from_numpy(self.lat[i]).float() * SCALE
        if random.random() < 0.5:
            c = torch.from_numpy(self.aud[i])
        else:
            c = torch.from_numpy(self.txt[i, random.randrange(5)])
        return z, c / c.norm()

ds = LatentDataset(lat, aud, txt)
loader = DataLoader(ds, batch_size = 64, shuffle = True, num_workers = 0, drop_last = True)

In [ ]:
T = 1000
# AudioLDM / LDM "scaled linear" schedule (linear in sqrt(beta)). v3 models used linspace(1e-4, 0.02).
SCHEDULE, BETA_START, BETA_END = "scaled_linear", 0.0015, 0.0195
betas = (torch.linspace(BETA_START ** 0.5, BETA_END ** 0.5, T) ** 2).to(device)
alphas = 1 - betas
alpha_bar =  torch.cumprod(alphas, dim = 0)

def add_noise(x0, t):
    noise = torch.randn_like(x0)
    a = alpha_bar[t]
    a = a.view(-1, 1, 1, 1)
    noisy = torch.sqrt(a) * x0 + torch.sqrt(1 - a) * noise
    return noisy, noise

def time_embedding(t, dim = 32):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device = t.device) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim = 1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim = 128):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding = 1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.norm2 = nn.GroupNorm(32, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding = 1)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)

class Attention(nn.Module):
    def __init__(self, ch, heads = 8):
        super().__init__()
        self.norm = nn.GroupNorm(32, ch)
        self.attn = nn.MultiheadAttention(ch, heads, batch_first = True)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).flatten(2).transpose(1, 2)
        h, _ = self.attn(h, h, h)
        h = h.transpose(1, 2).reshape(B, C, H, W)
        return x + h

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cond_proj = nn.Linear(512, 128)
        self.inp = nn.Conv2d(4, 64, 3, padding = 1)
        self.block1 = ResBlock(64, 64)
        self.down1 = nn.Conv2d(64, 64, 3, stride = 2, padding = 1)
        self.block2 = ResBlock(64, 128)
        self.down2 = nn.Conv2d(128, 128, 3, stride = 2, padding = 1)
        self.block3 = ResBlock(128, 256)
        self.mid = ResBlock(256, 512)
        self.attn = Attention(512)
        self.up = nn.Upsample(scale_factor = 2)
        self.block4 = ResBlock(768, 256)
        self.block5 = ResBlock(384, 128)
        self.block6 = ResBlock(192, 64)
        self.out_norm = nn.GroupNorm(32, 64)
        self.out = nn.Conv2d(64, 4, 1)

    def forward(self, x, t, c):
        t_emb = time_embedding(t, dim = 128) + self.cond_proj(c)
        x = self.inp(x)
        d1 = self.block1(x, t_emb)
        h = self.down1(d1)
        d2 = self.block2(h, t_emb)
        h = self.down2(d2)
        d3 = self.block3(h, t_emb)
        h = self.mid(d3, t_emb)
        h = self.attn(h)
        h = torch.cat([h, d3], dim = 1)
        h = self.block4(h, t_emb)
        h = self.up(h)
        h = torch.cat([h, d2], dim = 1)
        h = self.block5(h, t_emb)
        h = self.up(h)
        h = torch.cat([h, d1], dim = 1)
        h = self.block6(h, t_emb)
        return self.out(nn.functional.silu(self.out_norm(h)))

model = UNet().to(device)
x = torch.randn(2, 4, 20, 64, device = device)
t = torch.randint(0, T, (2,), device = device)
c = torch.randn(2, 512, device = device)
print(model(x, t, c).shape)
print(sum(p.numel() for p in model.parameters()))

In [ ]:
# ---------------------------------------------------------------------------
# U-Net v4: bigger, built from AudioLDM's recipe (ADM-style U-Net).
#   - 128 base channels, x1/x2/x4 per level, 2 ResBlocks per level
#   - time and CLAP each go through a 2-layer MLP, then FiLM (scale + shift) every ResBlock
#   - self-attention (32 channels per head) at the two lower resolutions and the middle
#   - last conv of each block zero-initialised, so every block starts as the identity
# ---------------------------------------------------------------------------
def zero(m):
    for p in m.parameters():
        nn.init.zeros_(p)
    return m


class ResBlock4(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, dropout = 0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding = 1)
        self.emb = nn.Linear(emb_dim, 2 * out_ch)
        self.norm2 = nn.GroupNorm(32, out_ch)
        self.drop = nn.Dropout(dropout)
        self.conv2 = zero(nn.Conv2d(out_ch, out_ch, 3, padding = 1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.emb(F.silu(emb))[:, :, None, None].chunk(2, dim = 1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.drop(F.silu(h)))
        return self.skip(x) + h


class Attention4(nn.Module):
    def __init__(self, ch, head_ch = 32):
        super().__init__()
        self.norm = nn.GroupNorm(32, ch)
        self.attn = nn.MultiheadAttention(ch, ch // head_ch, batch_first = True)
        zero(self.attn.out_proj)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).flatten(2).transpose(1, 2)
        h, _ = self.attn(h, h, h, need_weights = False)
        return x + h.transpose(1, 2).reshape(B, C, H, W)


class Down(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride = 2, padding = 1)

    def forward(self, x):
        return self.conv(x)


class Up(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding = 1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor = 2, mode = "nearest"))


class UNet4(nn.Module):
    def __init__(self, base = 128, mult = (1, 2, 4), nres = 2, attn = (False, True, True), emb = 512, dropout = 0.0):
        super().__init__()
        self.base = base
        self.time_mlp = nn.Sequential(nn.Linear(base, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.cond_mlp = nn.Sequential(nn.Linear(512, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.inp = nn.Conv2d(4, base, 3, padding = 1)

        self.down = nn.ModuleList()
        skips, ch = [base], base
        for lvl, m in enumerate(mult):
            for _ in range(nres):
                blocks = [ResBlock4(ch, base * m, emb, dropout)]
                ch = base * m
                if attn[lvl]:
                    blocks.append(Attention4(ch))
                self.down.append(nn.ModuleList(blocks))
                skips.append(ch)
            if lvl != len(mult) - 1:                      # 20x64 -> 10x32 -> 5x16
                self.down.append(nn.ModuleList([Down(ch)]))
                skips.append(ch)

        self.mid = nn.ModuleList([ResBlock4(ch, ch, emb, dropout), Attention4(ch), ResBlock4(ch, ch, emb, dropout)])

        self.up = nn.ModuleList()
        for lvl, m in reversed(list(enumerate(mult))):
            for i in range(nres + 1):
                blocks = [ResBlock4(ch + skips.pop(), base * m, emb, dropout)]
                ch = base * m
                if attn[lvl]:
                    blocks.append(Attention4(ch))
                if lvl != 0 and i == nres:
                    blocks.append(Up(ch))
                self.up.append(nn.ModuleList(blocks))

        self.out = nn.Sequential(nn.GroupNorm(32, ch), nn.SiLU(), zero(nn.Conv2d(ch, 4, 3, padding = 1)))

    @staticmethod
    def run(blocks, h, emb):
        for b in blocks:
            h = b(h, emb) if isinstance(b, ResBlock4) else b(h)
        return h

    def forward(self, x, t, c):
        emb = self.time_mlp(time_embedding(t, self.base)) + self.cond_mlp(c)
        h = self.inp(x)
        hs = [h]
        for blocks in self.down:
            h = self.run(blocks, h, emb)
            hs.append(h)
        h = self.run(self.mid, h, emb)
        for blocks in self.up:
            h = self.run(blocks, torch.cat([h, hs.pop()], dim = 1), emb)
        return self.out(h)

model = UNet4().to(device)
print(model(torch.randn(2, 4, 20, 64, device = device), torch.randint(0, T, (2,), device = device),
            torch.randn(2, 512, device = device)).shape)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")

In [ ]:
NAME = "unet_v4"     # bigger U-Net (AudioLDM recipe), scaled-linear schedule, unit-std latents
EPOCHS = 40
LR, WARMUP, EMA_DECAY = 1e-4, 2000, 0.9999       # AudioLDM: lr 1e-4, 2000 warm-up steps; LDM EMA 0.9999
use_bf16 = torch.cuda.get_device_capability()[0] >= 8   # L4 / A100 yes, T4 no
print("bf16:", use_bf16)

model = UNet4().to(device)
opt = torch.optim.AdamW(model.parameters(), lr = LR, weight_decay = 0.0)
ema = copy.deepcopy(model).eval().requires_grad_(False)

# the server reads this to know the architecture, latent scale and schedule
json.dump({"arch": "v4", "scale": SCALE, "schedule": SCHEDULE, "beta_start": BETA_START, "beta_end": BETA_END},
          open(CKPT + f"{NAME}_ema_config.json", "w"))

# Resume after a disconnect: rerun cells 1-7 and it continues from the last saved epoch.
# (Saved every 5 epochs, without optimizer state, to keep Drive usage small; warm-up reruns on resume.)
state_path = CKPT + f"{NAME}_state.pt"
start = 0
if os.path.exists(state_path):
    st = torch.load(state_path, map_location = device)
    model.load_state_dict(st["model"])
    ema.load_state_dict(st["ema"])
    start = st["epoch"] + 1
    print("resuming from epoch", start)
step, warm = start * len(loader), 0

for epoch in range(start, EPOCHS):
    total = 0
    for z, clap in tqdm(loader):
        for g in opt.param_groups:
            g["lr"] = LR * min(1.0, (warm + 1) / WARMUP)
        z = z.to(device)
        c = clap.to(device)
        mask = torch.rand(c.shape[0], device = device) < 0.1        # 10% unconditional, for CFG
        c[mask] = 0
        t = torch.randint(0, T, (z.shape[0],), device = device)
        noisy, noise = add_noise(z, t)
        with torch.autocast("cuda", dtype = torch.bfloat16, enabled = use_bf16):
            pred = model(noisy, t, c)
        loss = F.mse_loss(pred.float(), noise)
        opt.zero_grad(set_to_none = True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        decay = min(EMA_DECAY, (1 + step) / (10 + step))             # EMA warm-up
        with torch.no_grad():
            for p_ema, p in zip(ema.parameters(), model.parameters()):
                p_ema.lerp_(p, 1 - decay)
        step, warm = step + 1, warm + 1
        total += loss.item()
    print(epoch, total / len(loader), flush = True)
    torch.save(ema.state_dict(), CKPT + f"{NAME}_ema.pt")
    if (epoch + 1) % 5 == 0:
        torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "epoch": epoch}, state_path)
    if (epoch + 1) % 10 == 0:
        torch.save(ema.state_dict(), CKPT + f"{NAME}_ema_{epoch + 1}.pt")
print(torch.cuda.max_memory_allocated() / 1e9, "GB")

In [ ]:
NAME = "unet_v4"
cfg = json.load(open(CKPT + f"{NAME}_ema_config.json"))
prompts = ["rain falling", "a dog barking", "birds singing", "people talking"]
with torch.inference_mode():
    t_in = proc(text = prompts, return_tensors = "pt", padding = True).to(device)
    c = clap_model.get_text_features(**t_in).pooler_output
    c = F.normalize(c, dim = 1)
c = c.repeat_interleave(4, dim = 0)
c_none = torch.zeros_like(c)
w = 3
N = c.shape[0]

ema = UNet4().to(device)
ema.load_state_dict(torch.load(CKPT + f"{NAME}_ema.pt", map_location = device))
ema.eval()
vae = Autoencoder().to(device)
vae.load_state_dict(torch.load(CKPT + "vae_gan2.pt", map_location = device))
vae.eval()

torch.manual_seed(0)   # same noise every run, so models compare fairly
x = torch.randn(N, 4, 20, 64, device = device)
ts = np.linspace(T - 1, 0, 50).round().astype(int)       # 50 DDIM steps, same as the website
with torch.no_grad():
    for i, t in enumerate(ts):
        t_batch = torch.full((2 * N,), int(t), device = device)
        eps_label, eps_none = ema(torch.cat([x, x]), t_batch, torch.cat([c, c_none])).chunk(2)
        eps = eps_none + w * (eps_label - eps_none)
        ab = alpha_bar[t]
        ab_prev = alpha_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device = device)
        x0 = (x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)
        x = torch.sqrt(ab_prev) * x0 + torch.sqrt(1 - ab_prev) * eps
    print("generated latent std:", x.std().item(), "(real ~1.0)")
    mel = vae.decoder(x / cfg["scale"]).clamp(-1, 1)
    m = ((mel + 1) / 2) * (MAX - MIN) + MIN
    wav = vocoder(m.squeeze(1))

fig, axes = plt.subplots(4, 4, figsize = (14, 10))
for i in range(N):
    ax = axes[i // 4, i % 4]
    ax.imshow(mel[i].squeeze().cpu().numpy(), origin = "lower", aspect = "auto", vmin = -1, vmax = 1)
    ax.set_title(f"{prompts[i // 4]} {i % 4 + 1}")
plt.tight_layout()
plt.show()

for i in range(N):
    print(prompts[i // 4], i % 4 + 1)
    display(Audio(wav[i].squeeze().cpu().numpy(), rate = 22050))